# 📈 Pipeline de Ingestão: Mercado Financeiro (Alpha Vantage ➔ Delta Bronze)

Este notebook demonstra a ingestão resiliente de dados de cotações diárias de ações da B3 (`PETR4`, `VALE3`, `ITUB4`, etc.) consumindo a API da **Alpha Vantage** e persistindo na camada **Bronze** em formato **Delta Lake** com governança no **Unity Catalog**.

In [ ]:
# 1. Configuração do Catálogo e Variáveis de Ambiente
CATALOG = "workspace"  # Catálogo Unity Catalog
BRONZE_SCHEMA = "bronze_api"
BRONZE_TABLE = "cotacoes_alpha"
ANALYTICS_SCHEMA = "analytics_api"

# Define catálogo e cria os schemas se não existirem
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{ANALYTICS_SCHEMA}")

print(f"✅ Schemas configurados com sucesso em: {CATALOG}")

In [ ]:
# 2. Função de Ingestão com Tratamento de Exceções e Auditoria
import os
import requests
import pandas as pd
from datetime import datetime

# Obtenha sua chave gratuita em: https://www.alphavantage.co/support/#api-key
API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY", "SUA_API_KEY_AQUI")
BASE_URL = "https://www.alphavantage.co/query"
TICKERS = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA"]

def buscar_cotacoes_diarias(symbol: str) -> pd.DataFrame:
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "apikey": API_KEY,
        "outputsize": "compact",
    }
    
    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    
    if "Time Series (Daily)" not in data:
        raise ValueError(f"Falha ao obter série para {symbol}: {data}")
        
    ts = data["Time Series (Daily)"]
    df = (
        pd.DataFrame.from_dict(ts, orient="index")
        .reset_index()
        .rename(columns={
            "index": "data",
            "1. open": "abertura",
            "2. high": "alta",
            "3. low": "baixa",
            "4. close": "fechamento",
            "5. volume": "volume",
        })
    )
    
    df["data"] = pd.to_datetime(df["data"]).dt.date
    df["abertura"] = df["abertura"].astype(float)
    df["alta"] = df["alta"].astype(float)
    df["baixa"] = df["baixa"].astype(float)
    df["fechamento"] = df["fechamento"].astype(float)
    df["volume"] = df["volume"].astype(float)
    df["ticker"] = symbol
    df["data_ingestao"] = datetime.utcnow()
    return df

In [ ]:
# 3. Execução em Lote e Gravação Incremental em Delta Lake
dfs = []
for ticker in TICKERS:
    try:
        print(f"Ingerindo {ticker}...")
        dfs.append(buscar_cotacoes_diarias(ticker))
    except Exception as e:
        print(f"Aviso para {ticker}: {e}")

if dfs:
    df_final = pd.concat(dfs, ignore_index=True)
    df_spark = spark.createDataFrame(df_final)
    full_table = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"
    
    df_spark.write.format("delta").mode("append").saveAsTable(full_table)
    print(f"✅ {len(df_final)} registros gravados na tabela Delta: {full_table}")
    display(spark.table(full_table).limit(10))

In [ ]:
# 4. Criação da View Analítica de Consumo (Databricks SQL)
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{ANALYTICS_SCHEMA}.vw_cotacoes_resumo AS
SELECT
    ticker,
    MAX(data)                 AS ultima_data,
    MAX(alta)                 AS maior_alta_no_periodo,
    MIN(baixa)                AS menor_baixa_no_periodo,
    ROUND(AVG(fechamento), 2) AS preco_medio_fechamento,
    SUM(volume)               AS volume_total_negociado,
    MAX(data_ingestao)        AS ultima_ingestao
FROM {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}
GROUP BY ticker;
""")

display(spark.sql(f"SELECT * FROM {CATALOG}.{ANALYTICS_SCHEMA}.vw_cotacoes_resumo ORDER BY ticker"))